# N02 — Residual Dynamics (v1.1 corrected extraction)
**PhysLLM · pilot notebook · GPT-2 and Pythia-160M**

## Purpose
Capture the tensor entering the first Transformer block and the output of every block using forward hooks. Record final LayerNorm output separately.

**Why this revision:** the prior pilot used `output_hidden_states=True`. The last returned state can be after final LayerNorm, while intermediate states are block outputs. That mixes final normalization into the apparent last layer update. This notebook hooks block boundaries directly.

These five prompts are for instrumentation/debugging only. They are not evidence for Main Evaluation or population-level claims.

## Protocol\nFor each model, `state 0` is the hidden tensor entering the first block; `states 1…L` are block outputs **before final LayerNorm**. The final LayerNorm output is recorded separately and excluded from block-update differences.\n\nRaw Frobenius norms are retained, but prompt-comparison plots use \(\|h\|_F/\sqrt{T}\) to reduce the direct effect of varying token count; \(\|h\|_F/\sqrt{TD}\) is also reported as per-element RMS. These rescalings do not make different prompts semantically matched.\n\n**Main Evaluation (separate):** use a prespecified corpus with N ≥ 100 independent texts, exclude this pilot set, and resample by text (10,000 text-level bootstrap replicates).

In [ ]:
%pip -q install -U transformers accelerate pandas matplotlib

In [ ]:
import os, sys, json, platform
from datetime import datetime, timezone
from pathlib import Path
import torch, pandas as pd, matplotlib.pyplot as plt, transformers

torch.manual_seed(2026)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
MODEL_IDS = {
    "pythia-160m": "EleutherAI/pythia-160m",
    "gpt2": "gpt2",
}
MODELS_TO_RUN = ["pythia-160m", "gpt2"]
DTYPE = torch.float32
MAX_LENGTH = 512
PROMPTS = [
    {"prompt_id":"P1_general", "style":"general_prose", "text":"The small bird flew across the quiet garden."},
    {"prompt_id":"P2_technical", "style":"technical_prose", "text":"The system records information from several independent sensors."},
    {"prompt_id":"P3_academic", "style":"academic", "text":"Physics-guided machine learning provides interpretable representations of deep neural networks."},
    {"prompt_id":"P4_question", "style":"question_dialogue", "text":"Could you please explain how quantum entanglement violates local realism?"},
    {"prompt_id":"P5_code_mixed", "style":"code_plus_natural_language", "text":'''def bubble_sort(arr):
    n = len(arr)
    for i in range(n - 1):
        for j in range(n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr

# Explain the time complexity of this function.'''}
]

## 1. Load models and locate exact block boundaries
Only the configured GPT-2 and GPT-NeoX/Pythia module paths are supported here. If the expected path is absent, the run stops rather than guessing.

In [ ]:
def load_causal_lm(model_id, dtype=torch.float32):
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype, low_cpu_mem_usage=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    return tokenizer, model, device


def resolve_block_and_final_norm(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers, final_norm, architecture = model.transformer.h, model.transformer.ln_f, "gpt2"
    elif hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        layers, final_norm, architecture = model.gpt_neox.layers, model.gpt_neox.final_layer_norm, "gpt_neox_pythia"
    else:
        raise RuntimeError("Unsupported architecture: expected GPT-2 transformer.h or Pythia gpt_neox.layers")
    if len(layers) < 1:
        raise RuntimeError("No Transformer blocks found")
    return layers, final_norm, architecture

## 2. Hook block input/output; final LayerNorm remains separate
Every hook is removed even if the forward pass fails. All captured states, final-normalized output, and logits are checked for finite values.

In [ ]:
@torch.inference_mode()
def extract_dynamics(tokenizer, model, device, prompt, max_length=512):
    layers, final_norm_module, architecture = resolve_block_and_final_norm(model)
    batch = tokenizer(prompt, return_tensors="pt", truncation=True,
                      max_length=max_length, add_special_tokens=True)
    batch = {k: v.to(device) for k, v in batch.items()}
    captured, handles = {}, []

    def first_pre_hook(module, args):
        if not args or not isinstance(args[0], torch.Tensor):
            raise RuntimeError("First block input tensor not found")
        captured["state_0"] = args[0].detach()
    handles.append(layers[0].register_forward_pre_hook(first_pre_hook))

    for i, layer in enumerate(layers):
        def block_post_hook(module, args, output, i=i):
            value = output[0] if isinstance(output, (tuple, list)) else output
            if not isinstance(value, torch.Tensor):
                raise RuntimeError(f"Block {i} output is not a tensor")
            captured[f"state_{i+1}"] = value.detach()
        handles.append(layer.register_forward_hook(block_post_hook))

    def final_norm_hook(module, args, output):
        value = output[0] if isinstance(output, (tuple, list)) else output
        if not isinstance(value, torch.Tensor):
            raise RuntimeError("Final LayerNorm output is not a tensor")
        captured["final_norm_output"] = value.detach()
    handles.append(final_norm_module.register_forward_hook(final_norm_hook))

    try:
        outputs = model(**batch, use_cache=False, output_hidden_states=False)
    finally:
        for handle in handles:
            handle.remove()

    n_blocks = len(layers)
    expected = [f"state_{i}" for i in range(n_blocks + 1)]
    missing = [key for key in expected if key not in captured]
    if missing or "final_norm_output" not in captured:
        raise RuntimeError(f"Missing hook outputs: {missing}; final norm captured={'final_norm_output' in captured}")

    states = [captured[f"state_{i}"].detach().to("cpu", dtype=torch.float64)[0]
              for i in range(n_blocks + 1)]
    final_state = captured["final_norm_output"].detach().to("cpu", dtype=torch.float64)[0]
    logits = outputs.logits.detach().to("cpu", dtype=torch.float64)
    n_tokens, hidden_dim = int(batch["input_ids"].shape[1]), int(states[0].shape[-1])
    rows = []

    for i, h in enumerate(states):
        if not torch.isfinite(h).all():
            raise FloatingPointError(f"NaN/Inf in block-boundary state {i}")
        raw = torch.linalg.vector_norm(h).item()
        per_tok = raw / max(n_tokens, 1) ** 0.5
        per_el = raw / max(n_tokens * hidden_dim, 1) ** 0.5
        if i < n_blocks:
            delta = states[i + 1] - h
            if not torch.isfinite(delta).all():
                raise FloatingPointError(f"NaN/Inf in block update {i}")
            d_raw = torch.linalg.vector_norm(delta).item()
            d_tok = d_raw / max(n_tokens, 1) ** 0.5
            d_el = d_raw / max(n_tokens * hidden_dim, 1) ** 0.5
            relative = d_raw / max(raw, 1e-12)
        else:
            d_raw = d_tok = d_el = relative = float("nan")
        rows.append({
            "layer_state_idx": i, "rho_state": i / max(n_blocks, 1),
            "n_tokens": n_tokens, "hidden_dim": hidden_dim,
            "hidden_norm_fro": raw, "hidden_norm_per_token_l2": per_tok,
            "hidden_rms_per_element": per_el,
            "hidden_mean": h.mean().item(), "hidden_var": h.var(unbiased=False).item(),
            "block_update_norm_fro_to_next": d_raw,
            "block_update_norm_per_token_l2": d_tok,
            "block_update_rms_per_element": d_el,
            "relative_block_update_norm": relative,
        })

    for name, tensor in [("final LayerNorm output", final_state), ("logits", logits)]:
        if not torch.isfinite(tensor).all():
            raise FloatingPointError(f"NaN/Inf in {name}")
    final_norm_fro = torch.linalg.vector_norm(final_state).item()
    meta = {
        "model_id": getattr(model.config, "_name_or_path", None),
        "tokenizer_id": getattr(tokenizer, "name_or_path", None),
        "architecture_path": architecture, "n_tokens": n_tokens,
        "hidden_dim": hidden_dim, "n_blocks": n_blocks,
        "n_block_boundary_states": len(states),
        "dtype": str(next(model.parameters()).dtype), "device": str(device),
        "last_block_output_norm_fro": torch.linalg.vector_norm(states[-1]).item(),
        "final_norm_output_norm_fro": final_norm_fro,
        "final_norm_output_norm_per_token_l2": final_norm_fro / max(n_tokens, 1) ** 0.5,
        "final_norm_vs_last_block_max_abs_diff": torch.max(torch.abs(final_state - states[-1])).item(),
        "logits_shape": list(outputs.logits.shape),
    }
    return pd.DataFrame(rows), meta

## 3. Run the five-prompt pilot
Checkpoint load failures are recorded as `BLOCKED`. Extraction errors stop the run rather than being silently counted as a pass.

In [ ]:
all_metrics, run_metadata, load_errors = [], [], []
for model_key in MODELS_TO_RUN:
    model_id = MODEL_IDS[model_key]
    print(f"\n=== {model_key}: {model_id} ===")
    try:
        tokenizer, model, device = load_causal_lm(model_id, DTYPE)
    except Exception as exc:
        load_errors.append({"model_key": model_key, "model_id": model_id,
                            "error_type": type(exc).__name__, "error": str(exc)})
        print("BLOCKED — checkpoint load failed:", repr(exc))
        continue
    for p in PROMPTS:
        print("Running", p["prompt_id"], p["style"])
        df, meta = extract_dynamics(tokenizer, model, device, p["text"], MAX_LENGTH)
        df.insert(0, "style", p["style"])
        df.insert(0, "prompt_id", p["prompt_id"])
        df.insert(0, "model_key", model_key)
        all_metrics.append(df)
        run_metadata.append({**meta, "model_key": model_key,
                             "prompt_id": p["prompt_id"], "style": p["style"],
                             "prompt_text": p["text"]})
    del tokenizer, model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

metrics_df = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
metadata_df, errors_df = pd.DataFrame(run_metadata), pd.DataFrame(load_errors)
print("Completed model-prompt runs:", len(metadata_df))
print("Blocked model loads:", len(errors_df))
display(metadata_df if not metadata_df.empty else pd.DataFrame())
if not errors_df.empty: display(errors_df)

## 4. Plot length-normalized state and block-update norms
Use normalized measures to compare prompts with different token counts. Raw Frobenius norms remain in the output table for reproducibility.

In [ ]:
if metrics_df.empty:
    print("No successful runs; resolve blocked loads first.")
else:
    for model_key, sub in metrics_df.groupby("model_key"):
        fig, ax = plt.subplots(figsize=(9, 4.5))
        for pid, g in sub.groupby("prompt_id"):
            ax.plot(g["rho_state"], g["hidden_norm_per_token_l2"], marker="o", label=pid)
        ax.set(title=f"{model_key}: block-boundary state norm (length-normalized)",
               xlabel="Normalized residual-state depth ρ", ylabel="||h_l||_F / sqrt(T)")
        ax.grid(True, alpha=.3); ax.legend(); plt.show()

        fig, ax = plt.subplots(figsize=(9, 4.5))
        for pid, g in sub.groupby("prompt_id"):
            g = g.dropna(subset=["block_update_norm_per_token_l2"])
            ax.plot(g["rho_state"], g["block_update_norm_per_token_l2"], marker="o", label=pid)
        ax.set(title=f"{model_key}: block-update norm (length-normalized)",
               xlabel="Normalized residual-state depth ρ", ylabel="||h_(l+1) - h_l||_F / sqrt(T)")
        ax.grid(True, alpha=.3); ax.legend(); plt.show()

    display(metadata_df[["model_key", "prompt_id", "n_tokens", "last_block_output_norm_fro",
                         "final_norm_output_norm_fro", "final_norm_output_norm_per_token_l2"]].round(5))

## 5. Inspect full measurements

In [ ]:
if not metrics_df.empty:
    display(metrics_df[["model_key", "prompt_id", "style", "layer_state_idx", "rho_state",
        "n_tokens", "hidden_dim", "hidden_norm_fro", "hidden_norm_per_token_l2",
        "hidden_rms_per_element", "block_update_norm_fro_to_next",
        "block_update_norm_per_token_l2", "relative_block_update_norm"]].round(6))

## 6. Interpretation guardrails
- A large block update is a result to investigate, not by itself evidence of chaos, a phase transition, or a physical law.
- Raw Frobenius norm scales with sequence length; use the normalized measures for prompt comparisons.
- Final LayerNorm output is a separate operation and is not included in the block-update difference.
- GPT-2 and Pythia have different architecture/activation scales; do not interpret raw norms as physical energies.
- Five prompts are pilot-only. No significance claims or population generalizations are justified.

## 7. Export outputs and run manifest
Copy or download this folder before the Colab runtime ends.

In [ ]:
OUT_DIR = Path("physllm_n02_outputs_v1_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)
if not metrics_df.empty:
    metrics_df.to_csv(OUT_DIR / "n02_pilot_layer_metrics.csv", index=False)
if not metadata_df.empty:
    metadata_df.to_json(OUT_DIR / "n02_pilot_run_metadata.json", orient="records", indent=2)
if not errors_df.empty:
    errors_df.to_csv(OUT_DIR / "n02_blocked_loads.csv", index=False)
summary = {
    "notebook": "N02_residual_dynamics_v1_1_corrected.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "models_requested": MODELS_TO_RUN,
    "models_completed": sorted(metadata_df["model_key"].unique().tolist()) if not metadata_df.empty else [],
    "completed_model_prompt_runs": int(len(metadata_df)),
    "blocked_model_loads": int(len(errors_df)),
    "state_capture": "first-block input plus each Transformer block output before final LayerNorm",
    "final_layer_norm": "recorded separately; excluded from block update norm",
    "prompt_length_normalization": "Frobenius norm divided by sqrt(T) and sqrt(T*D)",
    "interpretation": "Pilot-only descriptive measurements; not Main Evaluation evidence."
}
with open(OUT_DIR / "n02_run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
display(summary)
print("Output folder:", OUT_DIR.resolve())

## 8. N02 pilot checklist
- [ ] Checkpoint loads recorded; blocked runs not counted as passes.
- [ ] Number of block-boundary states equals `n_blocks + 1`.
- [ ] All block outputs and final LayerNorm output were captured separately.
- [ ] No NaN/Inf in states, final LayerNorm output, or logits.
- [ ] Prompt-length-normalized plots inspected.
- [ ] CSV and JSON metadata exported.

**Gate status: INCONCLUSIVE until the corrected run is reviewed.** This notebook does not by itself establish a physics hypothesis or generalization.